# Analysing a Bitcoin address

Sync one testnet4 address into a temporary database, then look at its transactions with pandas. Data comes from mempool.space's public Esplora server at one request a second; no key is needed.

In [ ]:
import tempfile
from pathlib import Path

from multichain_indexer import Indexer

ADDRESS = "mgdKMwnT4S5gbx55Uw3tRJchUSrHwAFjzu"
folder = tempfile.TemporaryDirectory()
indexer = Indexer("bitcoin", database=Path(folder.name) / "example.db")
result = indexer.sync(ADDRESS)
result

Exports carry the shared fields as columns and the `native` block as JSON, with every input and output. Fees are in sats and fee rates in sat/vB.

In [ ]:
import io
import json

import pandas as pd

frame = pd.read_csv(io.StringIO(indexer.export("csv", account=result.account)))
native = frame["native"].map(json.loads)
frame["fee_rate"] = native.map(lambda n: round(n["fee_rate"], 2))
frame["vsize"] = native.map(lambda n: n["vsize"])
frame["inputs"] = native.map(lambda n: len(n["inputs"]))
frame["outputs"] = native.map(lambda n: len(n["outputs"]))
frame[["time", "id", "fee", "fee_rate", "vsize", "inputs", "outputs"]]

In [ ]:
outputs = pd.DataFrame([out for n in native for out in n["outputs"]])
outputs.groupby("script_type")["value"].agg(["count", "sum"])

In [ ]:
indexer.metrics(days=3650, account=result.account).native

In [ ]:
indexer.close()
folder.cleanup()